In [ ]:
import re
from pyspark.sql.functions import col, current_timestamp, regexp_extract

# `catalog` and `filename` are job parameters. `filename` (e.g. "routes.txt") is the ONLY per-file
# input; the table, checkpoint, path filter, and lineage regex all derive from it, so one for_each
# task list of filenames drives every GTFS static file. Defaults keep interactive runs working;
# schemas/volumes are identical across dev and prod.
dbutils.widgets.text("catalog", "transport_vic_dev")
dbutils.widgets.text("filename", "stops.txt")
catalog  = dbutils.widgets.get("catalog")
filename = dbutils.widgets.get("filename")

table_name = filename.removesuffix(".txt")          # stops.txt -> stops, calendar_dates.txt -> calendar_dates
table_path = f"{catalog}.`01_bronze`.{table_name}"
base       = f"/Volumes/{catalog}/01_bronze/gtfs_data/"
checkpoint = f"/Volumes/{catalog}/01_bronze/_checkpoints/{table_name}"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.`01_bronze`._checkpoints")

# 01_extract_schedule_zip writes export_date=YYYYMMDD/<feed>/<filename> for every GTFS .txt file.
# pathGlobFilter narrows this run to just this file; the per-file checkpoint is what makes "new"
# mean new, so nothing has to be handed over from the extract task.
FEED_RE = rf"/gtfs_data/export_date=(\d{{8}})/([^/]+)/{re.escape(filename)}$"

bronze = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", checkpoint + "/schema")
      .option("cloudFiles.inferColumnTypes", False)              # bronze = raw strings; cast in silver
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # a feed with an extra column widens
      .option("header", True)                                    # feeds differ in columns; map by name, not position
      .option("pathGlobFilter", filename)                        # this run's file only
      .load(base)
      # input_file_name() is unsupported in UC, so feed and export come off _metadata.file_path.
      .withColumn("_source_file", col("_metadata.file_path"))
      .withColumn("export_date", regexp_extract(col("_metadata.file_path"), FEED_RE, 1))
      .withColumn("feed_id",     regexp_extract(col("_metadata.file_path"), FEED_RE, 2))
      .withColumn("_ingest_ts", current_timestamp())
)

# mergeSchema lets a widened source (a feed that adds a column) grow the target table. Pairs with
# schemaEvolutionMode above; the stream restarts once to pick the new column up, so the bronze task
# needs retries enabled in the job.
(bronze.writeStream
        .option("checkpointLocation", checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(table_path)
 )